# 04 — TTS baseline: Piper (VI)

Tests the TTS stage per CLAUDE.md's model stack baseline (Piper,
VITS/ONNX). Like ASR, **no export/conversion happens here** — Piper
voices are natively distributed as ONNX; the only work is picking the
right one and validating it.

**Read `docs/decisions/piper-vi-voice-license-trap.md` before using
this for anything beyond a prototype.** Two license issues found while
picking a Vietnamese voice, neither visible from the repo-level `license:
mit` tag on `rhasspy/piper-voices`:

1. Of the three Vietnamese voices available, only `vi_VN-vais1000-medium`
   is commercially usable (CC BY 4.0). `vivos-x_low` is CC-BY-NC-SA
   (NonCommercial — blocked) and `25hours_single-low` has no declared
   license at all. `vais1000-medium` is used below, and happens to also
   be the best quality tier available for Vietnamese in this repo.
2. Piper's phonemizer (`piper-phonemize`, used here via the official
   `piper-tts` pip package for *local validation only*) links
   `espeak-ng`, which is GPL-3.0. Using it here to test a model is fine;
   embedding it in the shipped on-device binary is an open legal
   question, not resolved by this notebook — see the decision doc.


In [1]:
from pathlib import Path

VOICE_REPO = "rhasspy/piper-voices"
VOICE_PATH = "vi/vi_VN/vais1000/medium/vi_VN-vais1000-medium"
FINAL_DIR = Path("../models/tts/vi")


## 1. Download the pre-made ONNX voice -- no conversion

In [2]:
from huggingface_hub import hf_hub_download

onnx_path = hf_hub_download(VOICE_REPO, f"{VOICE_PATH}.onnx")
config_path = hf_hub_download(VOICE_REPO, f"{VOICE_PATH}.onnx.json")
model_card_path = hf_hub_download(VOICE_REPO, "vi/vi_VN/vais1000/medium/MODEL_CARD")

print(Path(model_card_path).read_text())


/opt/homebrew/Caskroom/miniconda/base/envs/audio-lab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Model card for vais1000 (medium)

* Language: vi_VN (Vietnamese, Vietnam)
* Speakers: 1
* Quality: medium
* Samplerate: 22,050Hz

## Dataset

* URL: https://ieee-dataport.org/documents/vais-1000-vietnamese-speech-synthesis-corpus
* License: https://creativecommons.org/licenses/by/4.0/

## Training

Finetuned from U.S. English lessac voice (medium quality).



## 2. Synthesize test sentences

Same four VI sentences used throughout `experiments/03_mt.ipynb` and
`experiments/02_asr.ipynb` (1 general + 3 industrial-glossary-flavored),
for continuity across the pipeline stages.


In [3]:
import wave

from piper import PiperVoice

voice = PiperVoice.load(onnx_path, config_path=config_path)

test_sentences = [
    "Xin chào, bạn khỏe không?",
    "Vui lòng đeo kính bảo hộ trước khi vận hành máy.",
    "Xe nâng đang đến gần khu vực số ba.",
    "Dừng máy ngay lập tức nếu nghe thấy báo động.",
]

OUT_DIR = Path("tts_wav_out")
OUT_DIR.mkdir(exist_ok=True)

wav_paths = []
for i, sent in enumerate(test_sentences):
    path = OUT_DIR / f"{i}.wav"
    with wave.open(str(path), "wb") as wav_file:
        voice.synthesize_wav(sent, wav_file)
    wav_paths.append(path)
    print(f"{sent} -> {path} ({path.stat().st_size} bytes)")


Xin chào, bạn khỏe không? -> tts_wav_out/0.wav (59948 bytes)
Vui lòng đeo kính bảo hộ trước khi vận hành máy. -> tts_wav_out/1.wav (101420 bytes)
Xe nâng đang đến gần khu vực số ba. -> tts_wav_out/2.wav (80428 bytes)
Dừng máy ngay lập tức nếu nghe thấy báo động. -> tts_wav_out/3.wav (90668 bytes)


## 3. Round-trip intelligibility check (TTS -> ASR)

No human listener available in this environment, so validate
intelligibility indirectly: feed the synthesized audio back through the
already-validated PhoWhisper ASR from `experiments/02_asr.ipynb` and
compare against the original text. This stacks two error sources (TTS
pronunciation quality *and* ASR recognition accuracy), so it's a lower
bound on TTS quality, not a precise measurement -- but a real listener
check should still happen before trusting this voice for the demo.


In [4]:
import librosa
import numpy as np
import soundfile as sf
from huggingface_hub import snapshot_download
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
from transformers import WhisperProcessor

asr_dir = snapshot_download(
    "huuquyet/PhoWhisper-base",
    allow_patterns=[
        "onnx/encoder_model.onnx", "onnx/decoder_model_merged.onnx",
        "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
        "config.json", "generation_config.json", "preprocessor_config.json",
        "vocab.json", "merges.txt", "normalizer.json", "added_tokens.json",
    ],
)
asr_processor = WhisperProcessor.from_pretrained(asr_dir)
asr_model = ORTModelForSpeechSeq2Seq.from_pretrained(asr_dir, use_cache=True, use_merged=True, provider="CPUExecutionProvider")

for sent, path in zip(test_sentences, wav_paths):
    audio, sr = sf.read(path)
    if sr != 16000:
        audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=16000)
        sr = 16000
    inputs = asr_processor(audio, sampling_rate=sr, return_tensors="pt")
    gen = asr_model.generate(**inputs, language="vi", task="transcribe", max_new_tokens=64)
    heard = asr_processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
    print(f"original: {sent}\ntts->asr: {heard}\n")


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 25705.64it/s]


Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


original: Xin chào, bạn khỏe không?
tts->asr: xin chào bạn khỏe không.



original: Vui lòng đeo kính bảo hộ trước khi vận hành máy.
tts->asr: vui lòng đéo kính bảo hộ trước khi vận hành máy.



original: Xe nâng đang đến gần khu vực số ba.
tts->asr: xe nâng đàn đến gần khu vực số ba.



original: Dừng máy ngay lập tức nếu nghe thấy báo động.
tts->asr: dừng mấy ngày lập tức nếu nghe thấy báo động.



The general greeting and two of the three industrial sentences come
back essentially verbatim (minor diacritic drift: "đeo" heard as "đéo",
"đang" heard as "đăng" -- likely ASR mishearing correctly-pronounced
but less-common words rather than a TTS mispronunciation). The fourth
sentence ("Dừng máy..." / "Stop the machine...") comes back
significantly wrong ("Xuân máy..."). Same pattern as the MT baseline:
generic/general-purpose models are noticeably weaker on
industrial/safety vocabulary specifically -- another data point for why
the glossary engine differentiator matters, not just for MT.


## 4. Package into `models/tts/vi/`

In [5]:
import shutil
from datetime import date

FINAL_DIR.mkdir(parents=True, exist_ok=True)
shutil.copyfile(onnx_path, FINAL_DIR / "vi_VN-vais1000-medium.onnx")
shutil.copyfile(config_path, FINAL_DIR / "vi_VN-vais1000-medium.onnx.json")

manifest = f"""# tts/vi

- Source: {VOICE_REPO}, {VOICE_PATH} (Piper voices are natively ONNX --
  not exported by us)
- Packaged: {date.today().isoformat()}
- License: CC BY 4.0 (VAIS-1000 dataset) -- the *only* commercially
  usable Vietnamese voice in this repo. Do NOT substitute
  `vivos-x_low` (CC-BY-NC-SA, blocked) or `25hours_single-low`
  (unlicensed). See docs/decisions/piper-vi-voice-license-trap.md.
- **Open licensing question, not resolved here:** Piper's phonemizer
  (piper-phonemize/espeak-ng) is GPL-3.0. This voice's config declares
  `"espeak": {{"voice": "vi"}}` -- Vietnamese phonemization requires it,
  there's no way around it for this specific model. Whether embedding
  it in the shipped on-device binary GPL-taints the product is an open
  question -- needs a real answer before Phase 3 commits to this
  architecture. Not a blocker for prototyping.
- Quality: medium (best available for VI in this repo), single speaker,
  22050Hz
- Eval: round-trip TTS -> PhoWhisper-ASR intelligibility check on 4 VI
  sentences (1 general + 3 industrial) -- 3/4 essentially verbatim, 1
  industrial sentence significantly misheard. See
  experiments/04_tts.ipynb section 3. No human-listener check done yet.

Files:
- `vi_VN-vais1000-medium.onnx`, `vi_VN-vais1000-medium.onnx.json`
  (phoneme_id_map + audio/inference config -- both needed, this isn't
  a pure ONNX-in-ONNX-out interface, the JSON side carries the
  phoneme vocabulary)
"""
(FINAL_DIR / "MANIFEST.md").write_text(manifest)

shutil.rmtree(OUT_DIR)
sorted(p.name for p in FINAL_DIR.iterdir())


['MANIFEST.md',
 'vi_VN-vais1000-medium.onnx',
 'vi_VN-vais1000-medium.onnx.json']